In [0]:
spark.conf.set(
    "fs.azure.account.key.batteryhealthdatalake.dfs.core.windows.net", 
    "SLUYElCah4m+RAnXDvZ9kBs1bi8/jnqVbge/BKCs+BcfcWrF9o5cmjcuW6S7/xsRzwJknh9vrYxa+AStwWjTSA=="
)

In [0]:
from pyspark.sql import functions as F

# Load silver data table
# Silver layer path
silver_path = "abfss://battery-data@batteryhealthdatalake.dfs.core.windows.net/silver/"

df_silver = f"{silver_path}battery_health_table"
df = spark.read.format("delta").load(df_silver)

In [0]:
# explode logs 
df_gold_exploded = df.withColumn("individual_logs", F.explode(F.col("logs")))

df_gold_exploded.display()

Feature Engineering

In [0]:
df_gold_features = df_gold_exploded.withColumn("event_type", F.regexp_extract(F.col("individual_logs"), r"^([^:]+):", 1))\
                                   .withColumn("event_value", F.regexp_extract(F.col("individual_logs"), r"(\d+\.?\d*)", 1).cast("double"))\
                                   .withColumn("module_id", F.regexp_extract(F.col("individual_logs"), r"on (Module_[A-Z])", 1))
df_gold_features.display()

Data aggregation

In [0]:
display(df_gold_features.groupBy("event_type").count())

In [0]:
df_gold_daily_kpis = df_gold_features.groupBy(F.window("timestamp", "1 day").alias("day_window"))\
                                    .agg(F.avg("voltage").alias("avg_voltage"),
                                          F.stddev("voltage").alias("voltage_stability"),
                                          F.max("temperature").alias("peak_temp"),
                                          F.count(F.when(F.col("event_type") == "V_DROOP", 1)).alias("total_voltage_sags"),
                                          F.count(F.when(F.col("event_type") == "THERMAL_GRADIENT", 1)).alias("thermal_stress_events"),
                                          F.avg("event_value").alias("avg_event_impact"))\
                                    .withColumn("report_data", F.col("day_window.start")).orderBy("report_data").drop("day_window")


df_gold_daily_kpis.display()

In [0]:
gold_path = "abfss://battery-data@batteryhealthdatalake.dfs.core.windows.net/gold/daily_battery_kpis"

df_gold_daily_kpis.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(gold_path)

spark.sql("CREATE TABLE IF NOT EXISTS gold_daily_battery_kpis USING DELTA")

In [0]:
df_gold_daily_kpis.write.mode("overwrite").saveAsTable("gold_battery_health_summary")